# MLflow fundamentals

## Handling Runs

**Author:** Alekzandev

- `mlflow.start_run()`: Start a new MLflow run in an experiment.
- `mlflow.end_run()`:  Finish a run xperiment.
- `mlflow.active_run()` Returns the run which is **currently active now**.
- `mlflow.last_active_run()`: Gets the **most recent active** run.


### `mlflow.start_run()`

In [1]:
import mlflow

In [6]:
mlflow.set_tracking_uri('../experiments')
experiment_id = '164880423579085298'
run_id = '7c2e957bdfea47e6845e4b6d8568cfb2'

In [7]:
with mlflow.start_run(
    run_id=run_id,
    nested=False,
    tags={
        "mlflow.runName": "my_run_name",
        "mlflow.source.name": "my_source_name",
        "mlflow.source.type": "LOCAL",
        "mlflow.user": "alekzandev",
        "mlflow.source.git.commit": "b2e6d7d3688136b4009fc51882492d25be438e8c",
        "mlflow.source.git.branch": "develop",
    },
    description="""
    This run is for test my hipotesis about the best way to use MLflow
    """
    ):
    mlflow.log_param("my_param", 1)
    mlflow.log_metric("my_metric", 2)
    mlflow.log_artifact("1-IntroMLflow.ipynb")

MlflowException: Changing param values is not allowed. Param with key='my_param' was already logged with value='-1' for run ID='7c2e957bdfea47e6845e4b6d8568cfb2'. Attempted logging new value '1'.

The cause of this error is typically due to repeated calls
to an individual run_id event logging.

Incorrect Example:
---------------------------------------
with mlflow.start_run():
    mlflow.log_param("depth", 3)
    mlflow.log_param("depth", 5)
---------------------------------------

Which will throw an MlflowException for overwriting a
logged parameter.

Correct Example:
---------------------------------------
with mlflow.start_run():
    with mlflow.start_run(nested=True):
        mlflow.log_param("depth", 3)
    with mlflow.start_run(nested=True):
        mlflow.log_param("depth", 5)
---------------------------------------

Which will create a new nested run for each individual
model and prevent parameter key collisions within the
tracking store.

In [14]:
with mlflow.start_run(
    run_id=run_id,
    nested=True,
    parent_run_id='eee23e3b324a42f1a6b2a56199545b58',
    tags={
        "mlflow.runName": "el-kink-runner",
        "mlflow.source.name": "HandlingRuns.ipynb",
        "mlflow.source.type": "Project",
        "mlflow.user": "alekzandev",
        "mlflow.source.git.commit": "b2e6d7d3688136b4009fc51882492d25be438e8c",
        "mlflow.source.git.branch": "develop",
    },
    description="""
    This run is for test my hipotesis about the best way to use MLflow
    """
    ):
    mlflow.log_param("my_param", -1)
    mlflow.log_metric("my_metric", 20)
    mlflow.log_param('my_param3', 2)
    mlflow.log_artifact("1-IntroMLflow.ipynb")

In [13]:
with mlflow.start_run(
    experiment_id=experiment_id
    ):
    mlflow.log_param("my_param", -1)
    mlflow.log_metric("my_metric", 20)
    mlflow.log_param('my_param2', 2)
    mlflow.log_artifact("1-IntroMLflow.ipynb")

In [17]:
help(mlflow.start_run)

Help on function start_run in module mlflow.tracking.fluent:

start_run(run_id: Optional[str] = None, experiment_id: Optional[str] = None, run_name: Optional[str] = None, nested: bool = False, parent_run_id: Optional[str] = None, tags: Optional[Dict[str, Any]] = None, description: Optional[str] = None, log_system_metrics: Optional[bool] = None) -> mlflow.tracking.fluent.ActiveRun
    Start a new MLflow run, setting it as the active run under which metrics and parameters
    will be logged. The return value can be used as a context manager within a ``with`` block;
    otherwise, you must call ``end_run()`` to terminate the current run.
    
    If you pass a ``run_id`` or the ``MLFLOW_RUN_ID`` environment variable is set,
    ``start_run`` attempts to resume a run with the specified run ID and
    other parameters are ignored. ``run_id`` takes precedence over ``MLFLOW_RUN_ID``.
    
    If resuming an existing run, the run status is set to ``RunStatus.RUNNING``.
    
    MLflow sets a v